# Regresión Logística Binaria - Guía de Estudio
## Caso: Supervivencia Titanic

**Objetivo**: Predecir la variables **Sobrevivió** (0 = No, 1 = Sí)

---

## Librerías Necesarias

```bash
pip install pandas numpy matplotlib seaborn statsmodels scikit-learn scipy
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

---

## BLOQUE 1: LIMPIEZA Y TRANSFORMACIÓN

### Teoría

Antes de aplicar regresión logística, los datos deben estar limpios.

**Manejo de Valores Nulos**
- Técnicas: Media/Mediana (numéricas), Moda (categóricas)

**Codificación de Variables Categóricas**
- Sexo: 'male'/'female' → 0/1 (Label Encoding)
- Puerto: 'C', 'Q', 'S' → Variables dummy (One-Hot Encoding)

In [ ]:
# GENERAR DATOS SINTÉTICOS TITANIC
np.random.seed(42)
n = 891

sexo = np.random.choice(['male', 'female'], n, p=[0.65, 0.35])
edad = np.random.normal(30, 15, n)
edad = np.clip(edad, 0.1, 80)
clase = np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55])
tarifa_base = {1: 80, 2: 40, 3: 20}
tarifa = np.array([tarifa_base[c] + np.random.normal(0, 10) for c in clase])
puerto = np.random.choice(['C', 'Q', 'S'], n, p=[0.20, 0.10, 0.70])

prob_supervivencia = 0.3
prob_supervivencia += (sexo == 'female').astype(int) * 0.4
prob_supervivencia += (clase == 1).astype(int) * 0.25
prob_supervivencia += (clase == 2).astype(int) * 0.10
prob_supervivencia += (edad < 15).astype(int) * 0.15
prob_supervivencia = np.clip(prob_supervivencia, 0, 1)

sobrevivio = (np.random.random(n) < prob_supervivencia).astype(int)

df_titanic = pd.DataFrame({
    'Sexo': sexo,
    'Edad': np.round(edad, 1),
    'Clase': clase,
    'Tarifa': np.round(tarifa, 2),
    'Puerto': puerto,
    'Sobrevivio': sobrevivio
})

np.random.seed(42)
nulos_edad = np.random.choice(n, 50, replace=False)
df_titanic.loc[nulos_edad[:25], 'Edad'] = np.nan
nulos_tarifa = np.random.choice(n, 30, replace=False)
df_titanic.loc[nulos_tarifa[:15], 'Tarifa'] = np.nan

print('DATOS TITANIC - PRIMERAS FILAS')
print(df_titanic.head(10))
print('\nVALORES NULOS:')
print(df_titanic.isnull().sum())

In [ ]:
# LIMPIEZA: Imputación de Valores Faltantes
df_limpio = df_titanic.copy()

imputador_edad = SimpleImputer(strategy='median')
df_limpio['Edad'] = imputador_edad.fit_transform(df_limpio[['Edad']])

imputador_tarifa = SimpleImputer(strategy='mean')
df_limpio['Tarifa'] = imputador_tarifa.fit_transform(df_limpio[['Tarifa']])

print('VALORES NULOS DESPUÉS DE IMPUTACIÓN:')
print(df_limpio.isnull().sum())

In [ ]:
# TRANSFORMACIÓN: Codificación de Variables Categóricas
df_limpio['Sexo_Cod'] = (df_limpio['Sexo'] == 'female').astype(int)

df_puerto_dummies = pd.get_dummies(df_limpio['Puerto'], prefix='Puerto', drop_first=False)
df_limpio = pd.concat([df_limpio, df_puerto_dummies], axis=1)

df_final = df_limpio[['Sexo_Cod', 'Edad', 'Clase', 'Tarifa', 'Puerto_C', 'Puerto_Q', 'Puerto_S', 'Sobrevivio']]

print('DATAFRAME FINAL PREPARADO:')
print(df_final.head(10))

---

## BLOQUE 2: PRUEBA ESTADÍSTICA T-TEST

### Teoría

El T-Test evalúa si la media de una variable continua es diferente entre dos grupos.

**Hipótesis**:
- H₀: Las medias son iguales
- H₁: Las medias son diferentes

**Interpretación**:
- p < 0.05: La variable ES relevante
- p >= 0.05: La variable NO es relevante

In [ ]:
# T-TEST: Tarifa entre Sobrevivientes y No Sobrevivientes
tarifa_supervivientes = df_final[df_final['Sobrevivio'] == 1]['Tarifa']
tarifa_no_supervivientes = df_final[df_final['Sobrevivio'] == 0]['Tarifa']

print('MEDIA DE TARIFA:')
print('   - Sobrevivientes:', tarifa_supervivientes.mean())
print('   - No sobrevivientes:', tarifa_no_supervivientes.mean())

t_stat, p_value = stats.ttest_ind(tarifa_supervivientes, tarifa_no_supervivientes)

print('\nRESULTADOS DEL T-TEST:')
print('   - Estadístico t:', t_stat)
print('   - p-value:', p_value)

if p_value < 0.05:
    print('\n   p-value < 0.05: La variable Tarifa ES RELEVANTE')
else:
    print('\n   p-value >= 0.05: La variable Tarifa podría NO ser relevante')

In [ ]:
# T-Test para Edad
edad_supervivientes = df_final[df_final['Sobrevivio'] == 1]['Edad']
edad_no_supervivientes = df_final[df_final['Sobrevivio'] == 0]['Edad']

t_stat_edad, p_value_edad = stats.ttest_ind(edad_supervivientes, edad_no_supervivientes)

print('T-TEST: EDAD')
print('   - p-value:', p_value_edad)
if p_value_edad < 0.05:
    print('   La variable Edad ES RELEVANTE')

In [ ]:
# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_final, x='Sobrevivio', y='Tarifa', ax=axes[0], palette=['#e74c3c', '#27ae60'])
axes[0].set_xlabel('Sobrevivio (0=No, 1=Sí)')
axes[0].set_ylabel('Tarifa ($)')
axes[0].set_title('Tarifa por Supervivencia')

sns.boxplot(data=df_final, x='Sobrevivio', y='Edad', ax=axes[1], palette=['#e74c3c', '#27ae60'])
axes[1].set_xlabel('Sobrevivio (0=No, 1=Sí)')
axes[1].set_ylabel('Edad (años)')
axes[1].set_title('Edad por Supervivencia')

plt.tight_layout()
plt.show()

---

## BLOQUE 3: ENTRENAMIENTO E INTERPRETACIÓN (ODDS RATIO)

### Teoría

En regresión logística:
- log(p/(1-p)) = β₀ + β₁X₁ + β₂X₂ + ...

**Interpretación**:
- Coeficiente (β): Cambio en log-odds
- Odds Ratio (exp(β)): Multiplicador de odds

- exp(β) = 1: Sin efecto
- exp(β) > 1: Aumenta probabilidad
- exp(β) < 1: Disminuye probabilidad

In [ ]:
# ENTRENAR MODELO
X_log = df_final[['Sexo_Cod', 'Edad', 'Clase', 'Tarifa', 'Puerto_C', 'Puerto_Q', 'Puerto_S']]
y_log = df_final['Sobrevivio']

X_train, X_test, y_train, y_test = train_test_split(X_log, y_log, test_size=0.2, random_state=42, stratify=y_log)

modelo_log = LogisticRegression(max_iter=1000, random_state=42)
modelo_log.fit(X_train, y_train)

print('COEFICIENTES DEL MODELO:')
coef_df = pd.DataFrame({
    'Variable': X_log.columns,
    'Coeficiente': modelo_log.coef_[0],
    'Odds Ratio': np.exp(modelo_log.coef_[0])
})
print(coef_df)
print('\nIntercepto:', modelo_log.intercept_[0])

In [ ]:
# INTERPRETACIÓN DE ODDS RATIOS
print('INTERPRETACIÓN DE ODDS RATIOS:')
for i, var in enumerate(X_log.columns):
    beta = modelo_log.coef_[0][i]
    odds_ratio = np.exp(beta)
    print(f'  {var}: Coef={beta:.4f}, OR={odds_ratio:.4f}')

---

## BLOQUE 4: MATRIZ DE CONFUSIÓN

### Teoría

|  | Predicho No | Predicho Sí |
|---|---|---|
| Real No | TN | FP |
| Real Sí | FN | TP |

**Métricas**:
- **Precision**: TP/(TP+FP)
- **Recall**: TP/(TP+FN)
- **F1-Score**: Media armónica
- **Accuracy**: (TP+TN)/Total

In [ ]:
# MATRIZ DE CONFUSIÓN
y_pred = modelo_log.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

print('MATRIZ DE CONFUSIÓN:')
print('              Predicho')
print('              No    Sí')
print(f'Real No   [{cm[0,0]:4d}  {cm[0,1]:4d}]')
print(f'Real Sí  [{cm[1,0]:4d}  {cm[1,1]:4d}]')

print('\n' + classification_report(y_test, y_pred, target_names=['No Sobrevivió', 'Sobrevivió']))

In [ ]:
# Visualización Matriz de Confusión
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Sobrevivió', 'Sobrevivió'],
            yticklabels=['No Sobrevivió', 'Sobrevivió'])
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusión')
plt.tight_layout()
plt.show()

---

## BLOQUE 5: CURVA ROC Y AUC

### Teoría

**Curva ROC**: Compara TPR vs FPR en diferentes umbrales.

**AUC**:
- 0.5: Modelo aleatorio
- 1.0: Perfecto
- > 0.8: Excelente

In [ ]:
# CURVA ROC Y AUC
y_proba = modelo_log.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

print('AUC (Área Bajo la Curva):', roc_auc)
if roc_auc >= 0.8:
    print('   Muy buena capacidad de discriminación')

In [ ]:
# Visualización Curva ROC
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#3498db', lw=2, label=f'Curva ROC (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Modelo Aleatorio')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## CONCLUSIÓN: Impacto en Decisiones de Negocio

| Métrica | Interpretación |
|--------|--------------|
| AUC | Capacidad de discriminación |
| Accuracy | Porcentaje correcto |
| Precision | Confiabilidad predicciones positivas |
| Recall | Encontrar todos los positivos |

**Recomendación**:
- Usar el modelo para identificar pasajeros de alto riesgo
- Enfocar recursos en: mujeres, niños, primera clase
- NO usar para decisiones sin supervisión humana